In [ ]:
# Implementation of LDA on the IMDB 5000 dataset

In [ ]:
from __future__ import absolute_import, division, print_function
import multiprocessing, os, pprint, re
import nltk
import gensim.models.word2vec as word2vec
import sklearn.manifold
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re, string

In [ ]:
pd.set_option('display.max_columns', None)
df = pd.read_csv('movie_metadata.csv')

In [ ]:
df = df.drop(df.movie_imdb_link.name, 1)
def trim_and_remove_spaces(raw):
    if pd.isnull(raw) == False:
        return raw.strip().replace(' ', '_')

df.director_name = df.director_name.apply(trim_and_remove_spaces)
df.actor_2_name = df.actor_2_name.apply(trim_and_remove_spaces)

df.actor_1_name = df.actor_1_name.apply(trim_and_remove_spaces)
df.movie_title = df.movie_title.apply(trim_and_remove_spaces)
df.actor_3_name = df.actor_3_name.apply(trim_and_remove_spaces)
df.language = df.language.apply(trim_and_remove_spaces)
df.country = df.country.apply(trim_and_remove_spaces)

In [ ]:
def descretize3(df, col):
    
    if(df[col].dtype == np.float64 or df[col].dtype == np.int64):
        df[col] = pd.qcut(df[col], 3, labels=['low', 'medium', 'high'], retbins=False)
        df[col] = df[col].apply(lambda x : (str(x)).replace(" ", ""))
        
def descretize2(df, col):
    
    if(df[col].dtype == np.float64 or df[col].dtype == np.int64):
        df[col] = pd.qcut(df[col], 2, labels=['low', 'high'], retbins=False)
        df[col] = df[col].apply(lambda x : (str(x)).replace(" ", ""))
        
def descretize10(df, col):
    
    if(df[col].dtype == np.float64 or df[col].dtype == np.int64):
        df[col] = pd.qcut(df[col], 3, labels=['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'], retbins=False)
        df[col] = df[col].apply(lambda x : (str(x)).replace(" ", ""))

descretize3(df, df.director_facebook_likes.name)
descretize3(df, df.num_critic_for_reviews.name)
descretize3(df, df.duration.name)
descretize3(df, df.actor_3_facebook_likes.name)
descretize3(df, df.actor_1_facebook_likes.name)

descretize3(df, df.gross.name)
descretize3(df, df.num_voted_users.name)
descretize3(df, df.cast_total_facebook_likes.name)
descretize3(df, df.num_user_for_reviews.name)
descretize3(df, df.budget.name)

descretize3(df, df.actor_2_facebook_likes.name)
descretize3(df, df.imdb_score.name)
descretize2(df, df.facenumber_in_poster.name)
descretize3(df, df.title_year.name)
descretize3(df, df.aspect_ratio.name)
descretize2(df, df.movie_facebook_likes.name)

In [ ]:
from sklearn.utils import shuffle
df = shuffle(df)
df

In [ ]:
raw_sentences = []
for a in df.iterrows():
    words = []    
    for b in a[1].iteritems():
        if b[0] == 'genres':
            for g in str(b[1]).split("|"):
                words.append('genre')
                words.append(trim_and_remove_spaces(g.lower()))
        elif b[0] == 'plot_keywords':
            for p in str(b[1]).split("|"):
                words.append('plot')
                words.append(trim_and_remove_spaces(p.lower()))
        elif b[0] in ['director_name', 'actor_2_name', 'actor_1_name', 'actor_3_name']:
            words.append(str(b[0]))
            words.append(str(b[1]))
        else:
            words.append(str(b[0]) + "_" + str(b[1]))
    raw_sentences.append(' '.join(words))

In [ ]:
def sentence_to_wordlist(raw):
    words = raw.split()
    return words

tokenized_sentences = []
for raw_sentence in raw_sentences:
    if len(raw_sentence) > 0:
        tokenized_sentences.append(sentence_to_wordlist(raw_sentence))

In [ ]:
token_count = sum([len(sentence) for sentence in tokenized_sentences])
print('Total token count = {0:,}'.format(token_count))

In [ ]:
tokenized_sentences[51]

In [ ]:
# Hyperparameters and training

num_features = 128
min_word_count = 1
num_workers = multiprocessing.cpu_count()
context_size = 10
downsampling = 1e-3
seed = 123

model = word2vec.Word2Vec(
    tokenized_sentences,
    sg=0, # for CBOW
    seed=seed,
    workers=num_workers,
    size=num_features,
    min_count=min_word_count,
    window=context_size,
    sample=downsampling,
    iter=10
)

In [ ]:
# Compress the model to 2D
tsne = sklearn.manifold.TSNE(n_components=2, random_state=0)
all_word_vectors_matrix = model.wv.syn0
all_word_vectors_matrix_2d = tsne.fit_transform(all_word_vectors_matrix)

In [ ]:
colors = {'genre': 1, 'plot': 1, 'director':3}
points = pd.DataFrame(
    [
        (word, coords[0], coords[1])
        for word, coords in [
            (word, all_word_vectors_matrix_2d[model.wv.vocab[word].index])
            for word in model.wv.vocab
        ]
    ],
    columns=["word", "x", "y"]
)

%pylab inline



In [ ]:
points['color'] = 0

In [ ]:
sns.set_context("notebook", font_scale=1.1)
sns.set_style("ticks")

sns.lmplot('x', 'y',
           data=points,
           fit_reg=False,
           scatter_kws={"marker": "D",
                        "s": 10})
plt.title('t-SNE')
plt.xlabel('X')
plt.ylabel('Y')

In [ ]:
def plot_region(x_bounds, y_bounds):
    slice = points[
        (x_bounds[0] <= points.x) &
        (points.x <= x_bounds[1]) & 
        (y_bounds[0] <= points.y) &
        (points.y <= y_bounds[1])
    ]
    
    ax = slice.plot.scatter("x", "y", s=35, figsize=(10, 8))
    for i, point in slice.iterrows():
        ax.text(point.x + 0.005, point.y + 0.005, point.word, fontsize=11)

In [ ]:
plot_region(x_bounds=(0.0, 0.2), y_bounds=(0.0, 0.2))

In [ ]:
model.wv.

In [ ]:
def likeBradPitt(v):
    if not v:
        return False
    return str(v) != 'Brad_Pitt' and model.wv.similarity(v, 'Brad_Pitt') > 0.8

def likeToken(v, token):
    try:
        return model.wv.similarity(v, token) > 0.5
    except:
        return False

In [ ]:
df[(df.movie_title.apply(lambda y: likeToken('movie_title_' + y, 'comedy'))) & (df.actor_1_name.apply(lambda x : likeBradPitt(x)))]